In [ ]:
import requests
import pandas as pd
import numpy as np
import os
import joblib
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("FOOTBALL_API_KEY")

In [ ]:
# Fetch World Cup 2026 standings from football-data.org API
headers = {"X-Auth-Token": API_KEY}
BASE_URL = "https://api.football-data.org/v4"

url_standings = f"{BASE_URL}/competitions/WC/standings?season=2026"
print(f"Fetching: {url_standings}")

response = requests.get(url_standings, headers=headers)
print(f"Status Code: {response.status_code}")

if response.status_code == 200:
    data_standings = response.json()
    print("Standings data received")
else:
    print(f"Error: {response.text}")

In [ ]:
# Extract team stats from standings
teams_2026 = []

for standing in data_standings.get("standings", []):
    for row in standing.get("table", []):
        team = row.get("team", {})
        teams_2026.append({
            "country_name": team.get("name"),
            "country_code": team.get("tla"),
            "matches_played": row.get("playedGames", 0),
            "wins": row.get("won", 0),
            "draws": row.get("draw", 0),
            "losses": row.get("lost", 0),
            "goals_for": row.get("goalsFor", 0),
            "goals_against": row.get("goalsAgainst", 0),
            "goal_difference": row.get("goalDifference", 0),
            "points": row.get("points", 0)
        })

df_current = pd.DataFrame(teams_2026)
print(f"Teams extracted: {len(df_current)}")
df_current.head()

In [ ]:
# Clean team names - remove HTML artifacts like 'rn">'
def clean_team_name(name):
    if pd.isna(name):
        return name
    if 'rn">' in name:
        return name.split('rn">')[-1]
    return name

df_current["country_name"] = df_current["country_name"].apply(clean_team_name)
print(f"Cleaned {len(df_current)} team names")

In [ ]:
# Load historical World Cup matches data
df_matches = pd.read_csv("../data/raw/WorldCupMatches.csv")
print(f"Historical matches: {len(df_matches)}")

In [ ]:
# Mapping: historical name -> name used in 2026 API data
NAME_MAPPING = {
    "Germany FR": "Germany",
    "USA": "United States",
    "Soviet Union": "Russia",
    'rn">Germany': "Germany",
    'rn">Republic of Ireland': "Republic of Ireland",
    'rn">Serbia and Montenegro': "Serbia and Montenegro",
    'rn">Trinidad and Tobago': "Trinidad and Tobago"
}

In [ ]:
# Calculate historical statistics for each team
all_historic_teams = set()
for col in ["Home Team Name", "Away Team Name"]:
    all_historic_teams.update(df_matches[col].dropna().unique())

# Map historical names to current names
def map_team_name(name):
    return NAME_MAPPING.get(name, name)

df_matches["Home Team Name"] = df_matches["Home Team Name"].apply(map_team_name)
df_matches["Away Team Name"] = df_matches["Away Team Name"].apply(map_team_name)

# Aggregate stats per team
historical_stats = []

for team_name in all_historic_teams:
    mapped_name = map_team_name(team_name)
    home = df_matches[df_matches["Home Team Name"] == mapped_name]
    away = df_matches[df_matches["Away Team Name"] == mapped_name]
    
    matches = len(home) + len(away)
    goals_for = home["Home Team Goals"].sum() + away["Away Team Goals"].sum()
    goals_against = home["Away Team Goals"].sum() + away["Home Team Goals"].sum()
    wins = (home["Win conditions"].str.contains(mapped_name, na=False)).sum() + \
           (away["Win conditions"].str.contains(mapped_name, na=False)).sum()
    
    historical_stats.append({
        "country_name": mapped_name,
        "matches_played": matches,
        "goals_for": goals_for,
        "goals_against": goals_against,
        "wins": wins,
        "draws": matches - wins - (len(home["Win conditions"].dropna()) + len(away["Win conditions"].dropna()) - wins),
        "losses": len(home) + len(away) - wins - (matches - wins - (len(home["Win conditions"].dropna()) + len(away["Win conditions"].dropna()) - wins))
    })

df_historical_agg = pd.DataFrame(historical_stats)
print(f"Historical stats calculated for {len(df_historical_agg)} teams")

In [ ]:
# Merge current 2026 data with historical stats
df_final = df_current.merge(
    df_historical_agg,
    on="country_name",
    how="left",
    suffixes=("_current", "_historical")
)

# Fill NaN with current values
for col in ["matches_played", "goals_for", "goals_against", "wins", "draws", "losses"]:
    if f"{col}_historical" in df_final.columns:
        df_final[col] = df_final[f"{col}_historical"].fillna(df_final[f"{col}_current"])
    else:
        df_final[col] = df_final[col].fillna(0)

print(f"Merged data: {len(df_final)} teams")

In [ ]:
# Confederation mapping for all 48 teams
CONFEDERATION_MAPPING = {
    # UEFA (16 teams)
    "France": "UEFA", "England": "UEFA", "Spain": "UEFA", "Italy": "UEFA",
    "Netherlands": "UEFA", "Germany": "UEFA", "Belgium": "UEFA",
    "Portugal": "UEFA", "Croatia": "UEFA", "Switzerland": "UEFA",
    "Denmark": "UEFA", "Austria": "UEFA", "Serbia": "UEFA",
    "Poland": "UEFA", "Scotland": "UEFA", "Slovenia": "UEFA",
    # CONMEBOL (6 teams)
    "Argentina": "CONMEBOL", "Brazil": "CONMEBOL", "Uruguay": "CONMEBOL",
    "Colombia": "CONMEBOL", "Ecuador": "CONMEBOL", "Paraguay": "CONMEBOL",
    # CONCACAF (4 teams)
    "United States": "CONCACAF", "Mexico": "CONCACAF", "Canada": "CONCACAF",
    "Jamaica": "CONCACAF",
    # AFC (8 teams)
    "Japan": "AFC", "Iran": "AFC", "South Korea": "AFC",
    "Australia": "AFC", "Saudi Arabia": "AFC", "Qatar": "AFC",
    "Iraq": "AFC", "Indonesia": "AFC",
    # CAF (9 teams)
    "Morocco": "CAF", "Senegal": "CAF", "Cameroon": "CAF",
    "Tunisia": "CAF", "Algeria": "CAF", "Egypt": "CAF",
    "Nigeria": "CAF", "South Africa": "CAF", "Cape Verde": "CAF",
    # OFC (2 teams)
    "New Zealand": "OFC", "Fiji": "OFC"
}

df_final["confederation"] = df_final["country_name"].map(CONFEDERATION_MAPPING).fillna("Other")

print("Confederations assigned")

In [ ]:
# Add placeholder for average_attendance (historical average)
df_final["average_attendance"] = 45000

# Calculate derived features
df_final["win_percentage"] = (df_final["wins"] / df_final["matches_played"].replace(0, 1) * 100).round(1)
df_final["goals_per_match"] = (df_final["goals_for"] / df_final["matches_played"].replace(0, 1)).round(2)

In [ ]:
# One-Hot Encoding for confederation
confederation_dummies = pd.get_dummies(df_final["confederation"], prefix="confederation", dtype=int)
df_final = pd.concat([df_final, confederation_dummies], axis=1)

In [ ]:
# Features in the exact order the model expects
FEATURES = [
    "matches_played", "goals_for", "goals_against", "wins", "draws", "losses",
    "average_attendance", "goals_per_match", "win_percentage", "goal_difference",
    "confederation_AFC", "confederation_CAF", "confederation_CONCACAF",
    "confederation_CONMEBOL", "confederation_OFC", "confederation_Other", "confederation_UEFA"
]

# Ensure all features exist
for feat in FEATURES:
    if feat not in df_final.columns:
        df_final[feat] = 0
        print(f"Warning: {feat} not found, set to 0")

In [ ]:
# Load and apply the trained scaler
scaler = joblib.load("../outputs/scaler.pkl")
X_2026_scaled = scaler.transform(df_final[FEATURES])

df_output = pd.DataFrame(X_2026_scaled, columns=FEATURES)
df_output.insert(0, "team", df_final["country_name"].values)

print(f"Scaled data shape: {df_output.shape}")

In [ ]:
# Save to CSV
output_path = "../data/raw/rankings_2026.csv"
df_output.to_csv(output_path, index=False)

print(f"Saved {len(df_output)} teams to {output_path}")
print(f"Columns: {list(df_output.columns)}")

In [ ]:
# Verify all required features are present
REQUIRED_FEATURES = [
    "matches_played", "goals_for", "goals_against", "wins", "draws", "losses",
    "average_attendance", "goals_per_match", "win_percentage", "goal_difference",
    "confederation_AFC", "confederation_CAF", "confederation_CONCACAF",
    "confederation_CONMEBOL", "confederation_OFC", "confederation_Other", "confederation_UEFA"
]

missing = [f for f in REQUIRED_FEATURES if f not in df_output.columns]
if missing:
    print(f"ERROR: Missing features: {missing}")
else:
    print("All required features present")